# Practical No-06: Retrieval-Augmented Generation (RAG) using a Pre-trained LLM

**Aim:** Build a RAG application over a domain-specific document collection, and compare answers **with retrieval (RAG)** vs **without retrieval (LLM-only)**.

**Domain:** Northbridge Technical University (NTU) student handbook (fictional, so the LLM cannot know the answers → hallucination is easy to demonstrate).

**Stack:** `sentence-transformers` (embeddings) · `ChromaDB` (vector database) · `Qwen2.5-Instruct` (pre-trained LLM, runs locally, no API key)

**Pipeline:** Documents → Clean → Chunk → Embed → Vector DB → Retrieve top-k → Prompt LLM with context → Answer

> Tip: Runtime → Change runtime type → GPU (T4) in Colab for faster generation.

In [ ]:
!pip install -q chromadb sentence-transformers transformers accelerate torch pypdf pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 0. CONFIG

In [ ]:
import os, re, glob, textwrap
import pandas as pd
import torch
import chromadb
from chromadb.utils import embedding_functions
from transformers import AutoTokenizer, AutoModelForCausalLM

DOCS_DIR   = "docs"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 384-d, fast, good for semantic search
LLM_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"                # use Qwen2.5-0.5B-Instruct if CPU is slow
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 400, 80, 3            # characters / chunks

## 1. DOCUMENT COLLECTION (creates sample docs if ./docs is empty)

In [ ]:
SAMPLE_DOCS = {
"academic_rules.txt": """
NTU Academic Regulations 2026

Attendance. A student must have a minimum of 80% attendance in each course to be eligible for the end-semester examination. Students with attendance between 70% and 80% may apply to the Dean of Academics for condonation, supported by a medical certificate. Below 70%, no condonation is allowed.

Promotion. A student is promoted to the next year if the number of backlogs (failed courses) does not exceed two. Students with more than two backlogs must repeat the year's failed courses during the summer term.

Examination form. The examination form must be submitted online by the 10th of the month preceding the exams. A late fee of Rs. 500 is charged for forms submitted within 5 days after the deadline. Forms are not accepted after that.

Grading. NTU uses a 10-point CGPA scale. A grade of DD (4 points) is the minimum passing grade. Re-evaluation of an answer sheet can be requested within 7 days of result declaration for a fee of Rs. 300 per course.
""",
"library_and_labs.txt": """
NTU Central Library Rules

Opening hours. The library is open from 8:30 AM to 9:00 PM on weekdays and 9:00 AM to 5:00 PM on Saturdays. It is closed on Sundays.

Borrowing. Undergraduate students may borrow up to 4 books for 21 days. Postgraduate students may borrow up to 6 books for 30 days. Books can be renewed once if no other student has reserved them.

Fines. A late fee of Rs. 7 per day per book is charged for overdue books. A lost book must be replaced or paid for at twice its listed price.

Laptop lending scheme. The library lends laptops to students from low-income families for a maximum of 14 days, extendable once by 7 days. A refundable deposit of Rs. 2000 is required.

Computer Labs. Labs are open until 8:00 PM. Food and drinks are strictly prohibited. Students must log in with their enrolment number.
""",
"hostel_and_welfare.txt": """
NTU Hostel and Student Welfare Handbook

Curfew. Hostel gates close at 9:30 PM for all residents. Late entry requires prior written permission from the warden. Three unauthorised late entries in a semester result in a warning letter to parents.

Visitors. Visitors are allowed only in the common room between 4:00 PM and 6:00 PM on Sundays. Overnight guests are not permitted.

Mess. The mess serves breakfast at 7:30 AM, lunch at 12:30 PM and dinner at 7:30 PM. A mess rebate is given if a student is absent for more than 5 consecutive days with prior intimation.

Grievance Cell. Students can raise complaints through the Grievance Cell, headed by Dr. Meera Kulkarni. The cell must acknowledge every complaint within 48 hours and resolve it within 15 working days. Complaints can be submitted at grievance@ntu.example or at Room 112, Admin Block.

Anti-ragging. NTU has a zero-tolerance policy. Ragging complaints are handled directly by the Anti-Ragging Committee chaired by the Vice-Chancellor.
""",
}

def ensure_docs():
    os.makedirs(DOCS_DIR, exist_ok=True)
    if not glob.glob(f"{DOCS_DIR}/*"):
        for name, text in SAMPLE_DOCS.items():
            open(f"{DOCS_DIR}/{name}", "w", encoding="utf-8").write(text.strip())
        print("Created sample documents in ./docs")

def load_documents():
    """Load .txt and .pdf files and clean the text."""
    docs = []
    for path in sorted(glob.glob(f"{DOCS_DIR}/*")):
        if path.endswith(".txt"):
            raw = open(path, encoding="utf-8").read()
        elif path.endswith(".pdf"):
            from pypdf import PdfReader
            raw = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
        else:
            continue
        docs.append({"source": os.path.basename(path), "text": clean(raw)})
    return docs

def clean(text):
    text = text.replace("\r", "")
    text = re.sub(r"[ \t]+", " ", text)          # collapse spaces
    text = re.sub(r"\n{3,}", "\n\n", text)       # collapse blank lines
    return text.strip()

In [ ]:
ensure_docs()
docs = load_documents()
for d in docs: print(d['source'], len(d['text']), 'chars')

Created sample documents in ./docs
academic_rules.txt 988 chars
hostel_and_welfare.txt 998 chars
library_and_labs.txt 822 chars


## 2. CHUNKING  (paragraph-aware sliding window with overlap)

Why: small chunks -> precise retrieval; overlap -> facts split at a boundary are not lost; paragraph boundaries keep each chunk semantically coherent.

In [ ]:
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks, cur = [], ""
    for p in paras:
        if len(cur) + len(p) + 1 <= size:
            cur = f"{cur}\n{p}".strip()
        else:
            if cur: chunks.append(cur)
            if len(p) > size:                      # very long paragraph -> sliding window
                step = size - overlap
                for i in range(0, len(p), step):
                    chunks.append(p[i:i + size])
                cur = ""
            else:
                tail = cur[-overlap:] if cur else ""   # overlap from previous chunk
                cur = f"{tail} {p}".strip()
    if cur: chunks.append(cur)
    return chunks

In [ ]:
# Inspect chunking
chs = chunk_text(docs[0]['text'])
print(len(chs), 'chunks from', docs[0]['source'])
for c in chs[:3]: print('---\n'+c)

4 chunks from academic_rules.txt
---
NTU Academic Regulations 2026
Attendance. A student must have a minimum of 80% attendance in each course to be eligible for the end-semester examination. Students with attendance between 70% and 80% may apply to the Dean of Academics for condonation, supported by a medical certificate. Below 70%, no condonation is allowed.
---
ation, supported by a medical certificate. Below 70%, no condonation is allowed. Promotion. A student is promoted to the next year if the number of backlogs (failed courses) does not exceed two. Students with more than two backlogs must repeat the year's failed courses during the summer term.
---
than two backlogs must repeat the year's failed courses during the summer term. Examination form. The examination form must be submitted online by the 10th of the month preceding the exams. A late fee of Rs. 500 is charged for forms submitted within 5 days after the deadline. Forms are not accepted after that.


## 3. EMBEDDINGS + VECTOR DATABASE (ChromaDB, cosine similarity)

In [ ]:
def build_vector_db(docs):
    ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
    client = chromadb.PersistentClient(path="chroma_store")
    try: client.delete_collection("ntu")
    except Exception: pass
    col = client.create_collection("ntu", embedding_function=ef, metadata={"hnsw:space": "cosine"})
    ids, texts, metas = [], [], []
    for d in docs:
        for i, c in enumerate(chunk_text(d["text"])):
            ids.append(f"{d['source']}-{i}"); texts.append(c)
            metas.append({"source": d["source"], "chunk": i})
    col.add(ids=ids, documents=texts, metadatas=metas)    # embeds + indexes
    print(f"Indexed {len(texts)} chunks from {len(docs)} documents")
    return col

def retrieve(col, query, k=TOP_K):
    r = col.query(query_texts=[query], n_results=k)
    return [{"text": t, "source": m["source"], "score": round(1 - d, 3)}
            for t, m, d in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

In [ ]:
collection = build_vector_db(docs)
for h in retrieve(collection, 'How many days can I borrow a laptop?'): print(h)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 11 chunks from 3 documents
{'text': 'erdue books. A lost book must be replaced or paid for at twice its listed price. Laptop lending scheme. The library lends laptops to students from low-income families for a maximum of 14 days, extendable once by 7 days. A refundable deposit of Rs. 2000 is required.\nComputer Labs. Labs are open until 8:00 PM. Food and drinks are strictly prohibited. Students must log in with their enrolment number.', 'source': 'library_and_labs.txt', 'score': 0.602}
{'text': 'ks for 30 days. Books can be renewed once if no other student has reserved them. Fines. A late fee of Rs. 7 per day per book is charged for overdue books. A lost book must be replaced or paid for at twice its listed price.', 'source': 'library_and_labs.txt', 'score': 0.305}
{'text': 'NTU Central Library Rules\nOpening hours. The library is open from 8:30 AM to 9:00 PM on weekdays and 9:00 AM to 5:00 PM on Saturdays. It is closed on Sundays.\nBorrowing. Undergraduate students may borrow 

## 4. PRE-TRAINED LLM + PROMPTING

In [ ]:
print("Loading LLM:", LLM_NAME)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, torch_dtype="auto", device_map="auto")

def generate(messages, max_new_tokens=150):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer_llm_only(q):
    return generate([
        {"role": "system", "content": "You are a helpful assistant. Answer briefly."},
        {"role": "user", "content": q}])

def answer_rag(col, q):
    hits = retrieve(col, q)
    context = "\n\n".join(f"[{i+1}] ({h['source']}) {h['text']}" for i, h in enumerate(hits))
    sys = ("You answer questions about Northbridge Technical University using ONLY the "
           "context provided. If the answer is not in the context, say "
           "'I don't know based on the provided documents.' Answer briefly and cite the source number.")
    user = f"Context:\n{context}\n\nQuestion: {q}"
    return generate([{"role": "system", "content": sys}, {"role": "user", "content": user}]), hits

Loading LLM: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
q = 'What is the minimum attendance required to sit for exams at NTU?'
print('LLM-only:', answer_llm_only(q))
rag_ans, hits = answer_rag(collection, q)
print('RAG     :', rag_ans)

LLM-only: NTU (Nanyang Technological University) does not have specific minimum attendance requirements for sitting for exams. However, it's generally expected that students attend classes regularly and complete their assignments on time to perform well in assessments. Students should check with their respective departments or academic advisors for any additional guidelines they may need to follow.
RAG     : Based on [1], the minimum attendance required to sit for exams at NTU is 80%.


## 5. EVALUATION: RAG vs LLM-only

Metrics: (a) Fact accuracy  - does the answer contain the ground-truth fact? (b) Grounded       - for RAG, is the gold fact present in retrieved chunks? (retrieval hit) (c) Hallucination  - answer is confident but fact is wrong / missing

In [ ]:
TEST_SET = [
    ("What is the minimum attendance required to sit for exams at NTU?", "80%"),
    ("What is the library late fee per day per book?", "Rs. 7"),
    ("What time do the hostel gates close?", "9:30"),
    ("How many backlogs are allowed to be promoted to the next year?", "two"),
    ("Who heads the NTU Grievance Cell?", "Meera Kulkarni"),
    ("What is the late fee for the exam form?", "500"),
    ("For how many days can a student borrow a laptop from the library?", "14"),
    ("What is the fee for re-evaluation of an answer sheet?", "300"),
    ("What is the capital of France?", "Paris"),   # control: general knowledge, no retrieval needed
]

def contains(text, fact):
    norm = lambda s: re.sub(r"[\s.,]", "", s.lower())
    return norm(fact) in norm(text)

def evaluate(col):
    rows = []
    for q, fact in TEST_SET:
        base = answer_llm_only(q)
        rag, hits = answer_rag(col, q)
        retrieved = any(contains(h["text"], fact) for h in hits)
        rows.append({
            "question": q, "gold": fact,
            "llm_only": base, "rag": rag,
            "llm_only_correct": contains(base, fact),
            "rag_correct": contains(rag, fact),
            "retrieval_hit": retrieved,
            "top_score": hits[0]["score"],
        })
    df = pd.DataFrame(rows)
    df.to_csv("rag_vs_llm_results.csv", index=False)
    print("\n=== RESULTS ===")
    for r in rows:
        print(f"\nQ: {r['question']}  (gold: {r['gold']})")
        print(f"  LLM-only [{'OK' if r['llm_only_correct'] else 'WRONG'}]: {textwrap.shorten(r['llm_only'], 200)}")
        print(f"  RAG      [{'OK' if r['rag_correct'] else 'WRONG'}] (retrieval hit={r['retrieval_hit']}, sim={r['top_score']}): {textwrap.shorten(r['rag'], 200)}")
    n = len(df)
    print("\n=== SUMMARY ===")
    print(f"LLM-only accuracy : {df.llm_only_correct.sum()}/{n}")
    print(f"RAG accuracy      : {df.rag_correct.sum()}/{n}")
    print(f"Retrieval hit rate: {df.retrieval_hit.sum()}/{n}")
    return df

# Out-of-scope test: RAG should refuse instead of hallucinating
def test_out_of_scope(col):
    q = "What is the tuition fee for the MBA programme at NTU?"
    print("\n=== OUT-OF-SCOPE TEST ===\nQ:", q)
    print("LLM-only:", answer_llm_only(q))
    print("RAG     :", answer_rag(col, q)[0])

In [16]:
df = evaluate(collection)
df[['question','gold','llm_only_correct','rag_correct','retrieval_hit','top_score']]


=== RESULTS ===

Q: What is the minimum attendance required to sit for exams at NTU?  (gold: 80%)
  LLM-only [WRONG]: NTU (Nanyang Technological University) does not have specific minimum attendance requirements for sitting for exams. However, it's generally expected that students attend classes regularly and [...]
  RAG      [OK] (retrieval hit=True, sim=0.78): Based on [1], the minimum attendance required to sit for exams at NTU is 80%.

Q: What is the library late fee per day per book?  (gold: Rs. 7)
  LLM-only [WRONG]: The exact amount of the library's late fee per day per book can vary depending on the specific library and its policies. It's best to check with your local library for their current policy.
  RAG      [OK] (retrieval hit=True, sim=0.529): The library late fee per day per book is Rs. 7. This information is explicitly stated in [2].

Q: What time do the hostel gates close?  (gold: 9:30)
  LLM-only [WRONG]: I'm sorry, but I don't have specific information about when ho

,question,gold,llm_only_correct,rag_correct,retrieval_hit,top_score
0,What is the minimum attendance required to sit...,80%,False,True,True,0.780
1,What is the library late fee per day per book?,Rs. 7,False,True,True,0.529
2,What time do the hostel gates close?,9:30,False,True,True,0.628
3,How many backlogs are allowed to be promoted t...,two,False,False,True,0.609
4,Who heads the NTU Grievance Cell?,Meera Kulkarni,False,True,True,0.539
5,What is the late fee for the exam form?,500,False,True,True,0.699
6,For how many days can a student borrow a lapto...,14,False,True,True,0.745
7,What is the fee for re-evaluation of an answer...,300,False,True,True,0.377
8,What is the capital of France?,Paris,True,False,False,0.087


In [14]:
test_out_of_scope(collection)


=== OUT-OF-SCOPE TEST ===
Q: What is the tuition fee for the MBA programme at NTU?
LLM-only: The tuition fee for the Master of Business Administration (MBA) programme at Nanyang Technological University (NTU) in Singapore can vary depending on several factors such as program level, duration, and whether you're an international student or local. As of my last update, the fees typically range from around $10,000 to over $25,000 per year for full-time students. For more accurate and up-to-date information, it's best to visit the official NTU website or contact their admissions office directly.
RAG     : I don't know based on the provided documents.


## 6. Demo: ask your own questions

Change the question and re-run to demonstrate the application.

In [15]:
def ask(q):
    ans, hits = answer_rag(collection, q)
    print("Q          :", q)
    print("RAG answer :", ans)
    print("LLM-only   :", answer_llm_only(q))
    print("\nRetrieved chunks:")
    for h in hits: print(f"  - ({h['source']}, sim={h['score']}) {textwrap.shorten(h['text'], 120)}")

ask("What time does the hostel gate close and what happens on late entry?")

Q          : What time does the hostel gate close and what happens on late entry?
RAG answer : The hostel gates close at 9:30 PM for all residents. Late entry requires prior written permission from the warden. Three unauthorised late entries in a semester result in a warning letter to parents.
LLM-only   : The exact closing times for hostels can vary, but typically, they do not allow late entries after their regular closing hours. Late arrivals may be allowed in some cases with prior notice or special arrangements made by the hostel staff. It's best to check the specific policies of the hostel you're planning to stay at.

Retrieved chunks:
  - (hostel_and_welfare.txt, sim=0.592) NTU Hostel and Student Welfare Handbook Curfew. Hostel gates close at 9:30 PM for all residents. Late entry [...]
  - (hostel_and_welfare.txt, sim=0.354) room between 4:00 PM and 6:00 PM on Sundays. Overnight guests are not permitted. Mess. The mess serves breakfast [...]
  - (library_and_labs.txt, sim=0.349) N

## 7. Observations and Conclusion (edit after running)

| Criterion | LLM-only | RAG |
|---|---|---|
| Accuracy on NTU-specific questions | Low (guesses generic values) | High (answers from retrieved text) |
| Hallucination | Frequent, stated confidently | Rare; refuses when context lacks the answer |
| Grounding / citations | None | Source chunk shown |
| General knowledge (France) | Correct | Correct / may refuse due to strict prompt |

**Findings:** RAG gives accurate, source-grounded answers on private or domain data that the LLM never saw in training. Failures of RAG are mostly **retrieval misses** (the right chunk is not in top-k) or a small LLM ignoring context.

**Limitations:** chunk size trade-off, top-k choice, small model capacity, no re-ranking.

**Improvements:** hybrid search (BM25 + vectors), cross-encoder re-ranking, larger LLM, metadata filters.